# Chapter 2 v2 — NB 02: ZNF175 carriers × phecode 389.4 (tinnitus) — carrier-case count, canonical method

**Goal:** re-count ZNF175 **carrier-cases** in v1 (11K) using the **canonical PheWAS phecode** (389.4 tinnitus, with control exclusions) from NB 01 — instead of our earlier raw-ICD definition — and see how it compares to our **4** and the PI's **8**.

Join: our ZNF175 v1 carriers (`chapter_2/results/06/carriers_v1.csv`, keyed by `GENO_ID`) × the phecode table (`phecodes_11k_with_genoid.csv`, also keyed by `GENO_ID`).

Reminder: **tinnitus = phecode 389.4** (389.2 is conductive HL). 389.4 maps from ICD-9 388.3x / ICD-10 H93.1x.

In [1]:
from pathlib import Path
import pandas as pd
BASE = Path("/project/hall/analysis/hearing-loss-genomics")
R2   = BASE / "analysis/chapter_2_v2/results"
C2   = BASE / "analysis/chapter_2/results"

PHE_COLS = ["GENO_ID","PT_ID","389.4","389","389.1","389.2"]
ph = pd.read_csv(R2/"phecodes_11k_with_genoid.csv", usecols=PHE_COLS, dtype=str)
carr = pd.read_csv(C2/"06/carriers_v1.csv").rename(columns={"IID":"GENO_ID"})
carr = carr[carr.carrier==1]
print("v1 ZNF175 carriers:", len(carr), "| phecode table participants:", len(ph))

v1 ZNF175 carriers: 34 | phecode table participants: 9322


## 1. Join carriers × phecodes and break down linkage

In [2]:
j = carr.merge(ph, on="GENO_ID", how="left")
in_table = j["PT_ID"].notna()
print(f"carriers total: {len(j)}")
print(f"  in phecode table (linked):   {int(in_table.sum())}")
print(f"  NOT in phecode table (unlinked / no phenotype): {int((~in_table).sum())}")

carriers total: 34
  in phecode table (linked):   27
  NOT in phecode table (unlinked / no phenotype): 7


## 2. Carrier-case counts by phecode (TRUE=case, FALSE=control, NA=excluded)

In [3]:
def brk(col, label):
    s = j[col].astype(str).str.upper()
    case = int((s=="TRUE").sum()); ctrl = int((s=="FALSE").sum())
    na   = int(((s!="TRUE")&(s!="FALSE")).sum())   # includes excluded + unlinked
    print(f"  {label:22s} (phecode {col:5s}): CASES={case}, controls={ctrl}, NA/excluded/unlinked={na}")
    return case

print("Among the", len(j), "ZNF175 v1 carriers:")
t  = brk("389.4","TINNITUS")
hl = brk("389","hearing loss")
snhl = brk("389.1","sensorineural HL")

Among the 34 ZNF175 v1 carriers:
  TINNITUS               (phecode 389.4): CASES=4, controls=21, NA/excluded/unlinked=9
  hearing loss           (phecode 389  ): CASES=4, controls=21, NA/excluded/unlinked=9
  sensorineural HL       (phecode 389.1): CASES=2, controls=21, NA/excluded/unlinked=11


## 2b. Sensitivity grid — does any reasonable definition move the carrier-cases off 4?
Test the axes that could plausibly explain the PI's 8: **date-vs-event counting** and **rule-of-2 vs rule-of-1**,
for tinnitus (389.4) and broader hearing-loss (389). Computed from raw diagnoses on the linked carriers.

In [4]:
FZ = Path("/static/PMBB/PMBB_Freeze17")
demo = pd.read_csv(FZ/"phenotype/PMBB_Geno_Demographics_Deidentified_012020.csv", dtype=str)\
         .drop_duplicates("GENO_ID")[["GENO_ID","PT_ID"]]
cpt = set(carr.merge(demo, on="GENO_ID", how="left").PT_ID.dropna())
diag = pd.read_csv(FZ/"phenotype/PMBB_Geno_Nonsensitive_Diagnosis_Deidentified_012020.csv",
                   header=None, usecols=[0,1,2,6], names=["PT_ID","CODE","VER","DATE"], dtype=str)
diag["vocabulary_id"] = diag["VER"].map({"ICD-9":"ICD9CM","ICD-10":"ICD10CM"})
pmap = pd.read_csv(R2/"phecode_map12.csv", dtype=str); roll = pd.read_csv(R2/"phecode_rollup_map.csv", dtype=str)
dm = (diag.merge(pmap, left_on=["vocabulary_id","CODE"], right_on=["vocabulary_id","code"], how="inner")
          .merge(roll, left_on="phecode", right_on="code", how="inner"))

def cc(pc, mode, thr):
    s = dm[(dm.phecode_unrolled==pc) & (dm.PT_ID.isin(cpt))]
    g = s.groupby("PT_ID")["DATE"].nunique() if mode=="dates" else s.groupby("PT_ID").size()
    return int((g>=thr).sum())

grid = pd.DataFrame([
    {"definition":"distinct-date >=2 (rule-of-2, canonical)", "tinnitus_389.4":cc("389.4","dates",2), "hearing_loss_389":cc("389","dates",2)},
    {"definition":"distinct-date >=1 (rule-of-1)",            "tinnitus_389.4":cc("389.4","dates",1), "hearing_loss_389":cc("389","dates",1)},
    {"definition":"event-count  >=2 (not deduped by date)",   "tinnitus_389.4":cc("389.4","events",2),"hearing_loss_389":cc("389","events",2)},
    {"definition":"event-count  >=1 (presence)",              "tinnitus_389.4":cc("389.4","events",1),"hearing_loss_389":cc("389","events",1)},
])
grid.to_csv(R2/"carrier_sensitivity_grid.csv", index=False)
print(grid.to_string(index=False))
u = int((dm[dm.phecode_unrolled.isin(["389.4","389"]) & dm.PT_ID.isin(cpt)]
           .groupby("PT_ID").size()>=1).sum())
print(f"\nTINNITUS(389.4) OR HL(389), rule-of-1 (union): {u} carriers")

                              definition  tinnitus_389.4  hearing_loss_389
distinct-date >=2 (rule-of-2, canonical)               4                 4
           distinct-date >=1 (rule-of-1)               5                 6
  event-count  >=2 (not deduped by date)               4                 5
             event-count  >=1 (presence)               5                 6

TINNITUS(389.4) OR HL(389), rule-of-1 (union): 6 carriers


## 2c. Annotation robustness — would ANNOVAR / REVEL / AlphaMissense add carrier-cases?
We used **VEP** for pLOF. Would a different annotator (ANNOVAR gene-based LOF, or damaging-missense via REVEL≥0.5 /
AlphaMissense) qualify more variants and lift the carrier-cases above 4? There is a hard **ceiling**: only a tinnitus
case carrying a **rare** (MAF≤0.1%) ZNF175 variant could *ever* qualify. That is **11 people** — our 4 pLOF plus 7
others. The table below shows what those 7 carry: if they are all synonymous/intron/UTR or **benign** missense
(REVEL≪0.5, AlphaMissense B), then **no annotator moves the count off 4**. REVEL/AlphaMissense come from our
dbNSFP ANNOVAR run (`results/04/…multianno.txt`); LOF class from VEP.

In [5]:
import subprocess
from collections import defaultdict
FZ    = Path("/static/PMBB/PMBB_Freeze17")
BCF   = "/appl/bcftools-1.21/bin/bcftools"
VZ    = BASE/"analysis/chapter_2/results/02/v1_znf175_strict_region.vcf.gz"
MULTI = BASE/"analysis/chapter_2/results/04/znf175_annovar.hg38_multianno.txt"

qv = pd.read_csv(BASE/"analysis/chapter_2/results/06/znf175_qualified_variants.csv", dtype=str)
qv["vid"] = qv.CHROM+":"+qv.POS+":"+qv.REF+":"+qv.ALT
cons = dict(zip(qv.vid, qv.Consequence)); isplof = dict(zip(qv.vid, qv.is_pLOF))

q = subprocess.run([BCF,"query","-f","[%SAMPLE\t%CHROM:%POS:%REF:%ALT\t%GT\n]",str(VZ)], capture_output=True, text=True).stdout
ac=defaultdict(int); an=defaultdict(int); salt=defaultdict(list)
for ln in q.split("\n"):
    if not ln: continue
    s,vid,gt=ln.split("\t"); a=gt.count("1"); an[vid]+=gt.count("0")+a; ac[vid]+=a
    if a>0: salt[s].append(vid)
maf = {v:(min(ac[v]/an[v],1-ac[v]/an[v]) if an[v] else 0) for v in an}
rare_ids = {v for v,m in maf.items() if 0<m<=0.001}

diag = pd.read_csv(FZ/"phenotype/PMBB_Geno_Nonsensitive_Diagnosis_Deidentified_012020.csv",
                   header=None, usecols=[0,1,2,6], names=["PT_ID","CODE","VER","DATE"], dtype=str)
tin = diag[diag.CODE.fillna("").str.match(r"^(388\.3|H93\.1)")].groupby("PT_ID")["DATE"].nunique()
tin_cases = set(tin[tin>=2].index)
demo = pd.read_csv(FZ/"phenotype/PMBB_Geno_Demographics_Deidentified_012020.csv", dtype=str).drop_duplicates("GENO_ID")[["GENO_ID","PT_ID"]]
p2g = defaultdict(list)
for gid,pid in zip(demo.GENO_ID, demo.PT_ID): p2g[pid].append(gid)

mv = pd.read_csv(MULTI, sep="\t", dtype=str); mv["vid"] = "19:"+mv.Start+":"+mv.Ref+":"+mv.Alt
revel = dict(zip(mv.vid, mv.REVEL_score)); amis = dict(zip(mv.vid, mv.AlphaMissense_pred))

rows=[]
for pid in tin_cases:
    for gid in p2g.get(pid,[]):
        for vid in salt.get(gid,[]):
            if vid in rare_ids:
                rows.append({"PT_ID":pid,"variant":vid,"MAF":round(maf[vid],6),"VEP":cons.get(vid,"NA"),
                             "is_pLOF":isplof.get(vid,"?"),"REVEL":revel.get(vid,"."),"AlphaMissense":amis.get(vid,".")})
rob = pd.DataFrame(rows).drop_duplicates().sort_values("is_pLOF", ascending=False)
rob.to_csv(R2/"annotation_robustness_tinnitus.csv", index=False)
ceiling = rob.PT_ID.nunique(); n_plof = rob[rob.is_pLOF=="True"].PT_ID.nunique()
print(f"tinnitus cases with a RARE ZNF175 variant (ceiling): {ceiling} | qualifying pLOF: {n_plof}")
print(f"the other {ceiling-n_plof} carry non-pLOF rare variants -> can any annotator upgrade them?\n")
print(rob.to_string(index=False))

tinnitus cases with a RARE ZNF175 variant (ceiling): 11 | qualifying pLOF: 4
the other 7 carry non-pLOF rare variants -> can any annotator upgrade them?

      PT_ID             variant      MAF                 VEP is_pLOF REVEL AlphaMissense
 1291558774    19:51588428:CA:C 0.000306  frameshift_variant    True     .             .
 1289754594    19:51588428:CA:C 0.000306  frameshift_variant    True     .             .
 1287154255 19:51587727:CAAAG:C 0.000131  frameshift_variant    True     .             .
 3514664072   19:51588214:CAG:C 0.000044  frameshift_variant    True     .             .
28001138357     19:51573321:A:G 0.000655 5_prime_UTR_variant   False     .             .
28001138357     19:51573425:T:C 0.000699  synonymous_variant   False     .             .
 1289800466     19:51581411:C:T 0.000306  synonymous_variant   False     .             .
 1290951865     19:51586967:C:T 0.000262  synonymous_variant   False     .             .
 1287242601     19:51581726:T:C 0.000611     

## 2d. v1 ↔ v2 — the **same 4 individuals**, and why v2 adds no new cases
The 4 carrier-cases are the *same people* in v1 (11K) and v2 (44K), even though the ID scheme changed
(UPENN → PMBB_ID). We can't match by participant ID, so we match by **variant fingerprint**: the rare frameshift
variants they carry, and the multiplicity of each. And we quantify whether "no new cases in v2" is surprising.

In [6]:
V2VCF = BASE/"analysis/chapter_2/results/02/v2_znf175_release_region.vcf.gz"
plof  = {v for v,f in isplof.items() if f=="True"}

# v1 fingerprint = the 4 pLOF tinnitus cases from §2c
v1fp = rob[rob.is_pLOF=="True"].variant.value_counts().to_dict()

# v2 carrier-cases (carrier & tinnitus) and their pLOF variants
a2  = pd.read_csv(BASE/"analysis/chapter_2/results/07/v2_analysis.csv")
cc2 = set(a2[(a2.carrier==1)&(a2.tinnitus==1)].PMBB_ID.astype(str))
q2  = subprocess.run([BCF,"query","-f","[%SAMPLE\t%CHROM:%POS:%REF:%ALT\t%GT\n]",str(V2VCF)],
                     capture_output=True, text=True).stdout
v2fp = defaultdict(int)
for ln in q2.split("\n"):
    if not ln: continue
    s,vid,gt = ln.split("\t")
    if s in cc2 and gt.count("1")>0 and vid in plof: v2fp[vid]+=1
v2fp = dict(v2fp)
print("v1 4-case variant fingerprint:", v1fp)
print("v2 4-case variant fingerprint:", v2fp)
print("IDENTICAL (same variants + multiplicity) ->", v1fp==v2fp, "=> the same 4 individuals\n")

# is 'no new cases' strange? expected carrier-cases under different ORs
n2=len(a2); ncarr2=int((a2.carrier==1).sum()); prev2=a2.tinnitus.mean()
rate_c=len(cc2)/ncarr2; rate_n=a2[a2.carrier==0].tinnitus.mean()
print(f"v2: {ncarr2} carriers | {len(cc2)} carrier-cases | tinnitus prev {prev2*100:.2f}%")
print(f"carrier tinnitus rate {rate_c*100:.1f}% vs non-carrier {rate_n*100:.2f}%  (RR {rate_c/rate_n:.1f}x -> real but modest)")
o=prev2/(1-prev2)
for OR,lab in [(1.0,"NULL"),(3.5,"regressed v2 OR"),(14.6,"Park discovery OR")]:
    exp=(OR*o)/(1+OR*o)*ncarr2
    print(f"  expected carrier-cases if OR={OR:>4}: {exp:4.1f}  [{lab}]")
print(f"  OBSERVED: {len(cc2)}  -> matches OR~3.5, above null, far below Park's ~16")

v1 4-case variant fingerprint: {'19:51588428:CA:C': 2, '19:51587727:CAAAG:C': 1, '19:51588214:CAG:C': 1}
v2 4-case variant fingerprint: {'19:51587727:CAAAG:C': 1, '19:51588214:CAG:C': 1, '19:51588428:CA:C': 2}
IDENTICAL (same variants + multiplicity) -> True => the same 4 individuals

v2: 69 carriers | 4 carrier-cases | tinnitus prev 1.97%
carrier tinnitus rate 5.8% vs non-carrier 1.97%  (RR 2.9x -> real but modest)
  expected carrier-cases if OR= 1.0:  1.4  [NULL]
  expected carrier-cases if OR= 3.5:  4.5  [regressed v2 OR]
  expected carrier-cases if OR=14.6: 15.7  [Park discovery OR]
  OBSERVED: 4  -> matches OR~3.5, above null, far below Park's ~16


## 3. Reconcile: our 4 (raw ICD) vs phecode 389.4 vs PI's 8

In [7]:
n_unlinked = int(j["PT_ID"].isna().sum())
summary = f'''# NB 02 — ZNF175 carrier-cases via canonical phecode (v1, 11K)

Carriers: {len(j)} (matches Park's ~35). In phecode table (linked): {int(in_table.sum())} | unlinked: {n_unlinked}.

## Carrier-cases by definition
- our earlier RAW ICD (388.3x/H93.1x, rule-of-2): 4
- **canonical PHECODE 389.4 (tinnitus, + control exclusions): {t}**
- phecode 389 (hearing loss, broader): {hl}
- phecode 389.1 (sensorineural HL): {snhl}

## Sensitivity grid (does any reasonable definition reach 8?)
- tinnitus 389.4 is **{t}** under BOTH distinct-date≥2 and event-count≥2 — event-vs-date counting does NOT move it.
- it reaches 5 only with rule-of-1 (presence); broadening to hearing-loss 389 gives 5 (event≥2) or 6 (rule-of-1);
  tinnitus-OR-HL rule-of-1 = {u}.
- **No definition on the linked carriers reaches 8** (max = {grid[["tinnitus_389.4","hearing_loss_389"]].max().max()}).

## Annotation robustness (§2c) — VEP vs ANNOVAR/REVEL/AlphaMissense
- ceiling = {ceiling} tinnitus cases carry a rare ZNF175 variant; only {n_plof} are qualifying pLOF.
- the other {ceiling-n_plof} carry synonymous / intron / UTR / **benign** missense (REVEL≪0.5, AlphaMissense B) —
  no annotator (ANNOVAR LOF, REVEL≥0.5, AlphaMissense) upgrades them. Carrier-cases are **robust to annotation tool**, not just counting rule.

## v1 ↔ v2 same individuals (§2d)
- the 4 carrier-cases are the SAME 4 people in v1 and v2 (variant fingerprint {v1fp} identical across freezes; IDs only recoded).
- "no new cases in v2" is NOT strange: carriers stay ~{rate_c/rate_n:.0f}x enriched (real, modest); observed 4 matches OR~3.5,
  sits above null (~1.4) and far below Park's OR~14.6 (which predicts ~16). Winner's curse made concrete —
  the discovery effect was anchored on these 4, and quadrupling the cohort added ~0 cases.

## Reading
- Canonical phecode tinnitus gives {t} carrier-cases (vs our raw-ICD 4) — robust; event-vs-date and the rollup do
  not change it.
- The gap to the PI's 8 is NOT closed by the phenotype definition. The remaining lever is the {n_unlinked} carriers
  with no phenotype linkage (their tinnitus status is unknown) — need the full PMBB ID map to assess them.

## Output
- znf175_carriers_phecode.csv (per-carrier phecode status)
- carrier_sensitivity_grid.csv (definition × carrier-case count)
'''
j.to_csv(R2/"znf175_carriers_phecode.csv", index=False)
(R2/"nb02_carrier_cases_phecode_summary.md").write_text(summary)
print(summary)

# NB 02 — ZNF175 carrier-cases via canonical phecode (v1, 11K)

Carriers: 34 (matches Park's ~35). In phecode table (linked): 27 | unlinked: 7.

## Carrier-cases by definition
- our earlier RAW ICD (388.3x/H93.1x, rule-of-2): 4
- **canonical PHECODE 389.4 (tinnitus, + control exclusions): 4**
- phecode 389 (hearing loss, broader): 4
- phecode 389.1 (sensorineural HL): 2

## Sensitivity grid (does any reasonable definition reach 8?)
- tinnitus 389.4 is **4** under BOTH distinct-date≥2 and event-count≥2 — event-vs-date counting does NOT move it.
- it reaches 5 only with rule-of-1 (presence); broadening to hearing-loss 389 gives 5 (event≥2) or 6 (rule-of-1);
  tinnitus-OR-HL rule-of-1 = 6.
- **No definition on the linked carriers reaches 8** (max = 6).

## Annotation robustness (§2c) — VEP vs ANNOVAR/REVEL/AlphaMissense
- ceiling = 11 tinnitus cases carry a rare ZNF175 variant; only 4 are qualifying pLOF.
- the other 7 carry synonymous / intron / UTR / **benign** missense (REVEL≪0.5, Alph